# Spam Message Classifier

## Question
Can we identify spam messages from their text?

## Plan
1. Explore labeled messages.
2. Train a simple model.
3. Evaluate it on messages it hasn't seen.
4. Investigate its mistakes.

## What matters most
Avoid marking legitimate messages as spam.
Missing some spam is acceptable.

I will examine spam precision and count legitimate messages
that are incorrectly flagged.

In [38]:
%pip install pandas scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
import pandas as pd
print("pandas is ready")

pandas is ready


In [40]:
messages = pd.read_csv("data/SMSSpamCollection", sep="\t", names=["label", "message"], quoting=3)
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [41]:
print("Total messages: ", len(messages))
print("\nMessages in each category:")
print(messages["label"].value_counts())

Total messages:  5574

Messages in each category:
label
ham     4827
spam     747
Name: count, dtype: int64


In [42]:
print("Missing values:")
print(messages.isna().sum())

print("\nRepeated message texts")
print(messages["message"].duplicated().sum())

Missing values:
label      0
message    0
dtype: int64

Repeated message texts
403


In [43]:
repeated = messages["message"].duplicated(keep=False)
messages.loc[repeated].sort_values("message").head(10)

,label,message
505,spam,+123 Congratulations - in this week's competit...
2124,spam,+123 Congratulations - in this week's competit...
2344,ham,1) Go to write msg 2) Put on Dictionary mode 3...
1373,ham,1) Go to write msg 2) Put on Dictionary mode 3...
2163,ham,1) Go to write msg 2) Put on Dictionary mode 3...
1050,spam,18 days to Euro2004 kickoff! U will be kept in...
2719,spam,18 days to Euro2004 kickoff! U will be kept in...
2044,spam,4mths half price Orange line rental & latest c...
389,spam,4mths half price Orange line rental & latest c...
1779,ham,7 wonders in My WORLD 7th You 6th Ur style 5th...


In [44]:
labels_per_message = messages.groupby("message")["label"].nunique()
print("Messages with conflicting labels:", (labels_per_message > 1).sum())

Messages with conflicting labels: 0


In [45]:
clean_messages = messages.drop_duplicates(subset="message").copy()

print("Original messages:", len(messages))
print("Unique messages:", len(clean_messages))

Original messages: 5574
Unique messages: 5171


The results could inflate performance as the algorithm is based on that exact same data, so it would get a lot more of the "test" data correct (it being the same). It has already seen those examples

In [46]:
print(clean_messages["label"].value_counts())

label
ham     4518
spam     653
Name: count, dtype: int64


In [47]:
from sklearn.model_selection import train_test_split

train_data, remaining_data = train_test_split(clean_messages,test_size = 0.4, random_state=42, stratify=clean_messages["label"])

print("Training messages:", len(train_data))
print("Remaining messages:", len(remaining_data))

Training messages: 3102
Remaining messages: 2069


In [48]:
validation_data, test_data = train_test_split(remaining_data, test_size=0.5, random_state = 42, stratify=remaining_data["label"])
print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Training: 3102
Validation: 1034
Test: 1035


In [49]:
baseline_accuracy = (validation_data["label"] == "ham").mean()

print(f"Always-ham validation accuracy: {baseline_accuracy:.1%}")

Always-ham validation accuracy: 87.4%


In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(train_data["message"])
X_validation = vectorizer.transform(validation_data["message"])
print("Trainig matrix shape:", X_train.shape)

Trainig matrix shape: (3102, 6462)


In [51]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

#Learn feature weights using the training messages and labels.
model.fit(X_train,train_data["label"])

#Predict labels for validation messages without seeing their answers.
validation_predictions = model.predict(X_validation)

print("Model trained and validation predictions created.")

Model trained and validation predictions created.


In [52]:
from sklearn.metrics import confusion_matrix

results = confusion_matrix(validation_data["label"],validation_predictions, labels=["ham", "spam"])

pd.DataFrame(results,index=["Actually ham", "Actually spam"],columns=["Predicted ham", "Predicted spam"])

,Predicted ham,Predicted spam
Actually ham,902,2
Actually spam,41,89


In [53]:
validation_review = validation_data.copy()
validation_review["prediction"] = validation_predictions
false_positives = validation_review[(validation_review["label"] == "ham") & (validation_review["prediction"] == "spam")]

for message in false_positives["message"]:
    print(message)
    print()

Sir, i am waiting for your call, once free please call me.

I (Career Tel) have added u as a contact on INDYAROCKS.COM to send FREE SMS. To remove from phonebook - sms NO to  &lt;#&gt;



In [54]:
word_weights = pd.Series(model.coef_[0],index=vectorizer.get_feature_names_out())

print("Class order:", model.classes_)
print("\nWeights for words in these messages:")
print(word_weights.reindex(["free", "call", "sms", "contact"]))

Class order: ['ham' 'spam']

Weights for words in these messages:
free       3.143296
call       4.007223
sms        1.164934
contact    1.484040
dtype: float64


Both false positives contained words with positive spam weights. This suggests the model can mistake ordinary or service-related language for spam. Because it uses individual words, it does not directly distinguish phrases such as "once free" from "free prize."